# Create Outer Join Datasets with Missingness Indicators

Rebuilds the 8 merged datasets using **outer join** instead of inner join.

**Why:** Inner join across 7 modalities left only 526 patients (from ~1098). Outer join recovers ~550 additional patients by filling missing modalities with 0 and adding a binary `MODALITY_missing` indicator per modality.

**Original MERGE/ folder is NOT modified.** New files go to `MERGE_outer/`.

**Missingness indicators added:**
- `RNA_missing`, `CNV_missing`, `MUT_missing`, `PROT_missing`, `METH_missing`, `MIRNA_missing`
- Value = 1 if patient had no data for that modality

**Restart kernel before running.**

## Paths & Setup

In [3]:
"""
CREATE OUTER JOIN DATASETS WITH MISSINGNESS INDICATORS

Strategy:
  - For each of the 8 datasets: re-merge all modalities using OUTER join
  - Missing values filled with 0 (median for continuous clinical features)
  - Add binary missingness indicator per modality: MODALITY_missing = 1 if absent
  - This expands sample size from ~526 (inner join) back to ~1000+

Why this may improve C-index:
  - More samples = better model generalization
  - Missingness pattern itself is informative (e.g. patients without protein data
    may differ systematically from those with it)
  - Avoids discarding ~550 patients who have partial data

Output: MERGE_outer/ (new folder — original MERGE/ untouched)

Script location: 01_Causal_feature_extraction/
"""

import pandas as pd
import numpy as np
from pathlib import Path

# ============================================================================
# PATHS
# ============================================================================

try:
    SCRIPT_DIR = Path(__file__).resolve().parent
except NameError:
    cwd = Path.cwd()
    SCRIPT_DIR = cwd if cwd.name == "MB" else cwd / "MB"
    SCRIPT_DIR = SCRIPT_DIR.parent  # back to 01_Causal_feature_extraction

BASE_DIR   = SCRIPT_DIR
MERGE_DIR  = BASE_DIR / "MERGE"          # original inner-join datasets (DO NOT MODIFY)
OUTPUT_DIR = BASE_DIR / "MERGE_outer"    # new outer-join datasets
OUTPUT_DIR.mkdir(exist_ok=True)

CLIN_FILE  = BASE_DIR / "Clinical" / "clinical_preprocessed_prefixed.csv"

OUTCOME_CANDIDATES = [
    BASE_DIR.parent / "data" / "outcome.csv",
    BASE_DIR / "RNA" / "preprocessed" / "outcome.csv",
]

print(f"Base dir:    {BASE_DIR}")
print(f"Input:       {MERGE_DIR}  (original — will NOT be modified)")
print(f"Output:      {OUTPUT_DIR}  (new outer-join datasets)")
print()

DATASET_NAMES = [
    "01_ultra_conservative", "02_conservative", "03_standard",
    "04_fdr_significant",    "05_balanced",     "06_correlation",
    "07_top_correlated",     "08_composite",
]

MODALITY_PREFIXES = {
    "RNA":   "RNA_",
    "CNV":   "CNV_",
    "MUT":   "MUT_",
    "PROT":  "PROT_",
    "METH":  "METH_",
    "MIRNA": "MIRNA_",
}

Base dir:    C:\Users\olegk\Desktop\Thesis_v3\01_Causal_feature_extraction
Input:       C:\Users\olegk\Desktop\Thesis_v3\01_Causal_feature_extraction\MERGE  (original — will NOT be modified)
Output:      C:\Users\olegk\Desktop\Thesis_v3\01_Causal_feature_extraction\MERGE_outer  (new outer-join datasets)



## Step 1 — Load Clinical Base

In [5]:
# ============================================================================
# STEP 1: LOAD CLINICAL (base — all patients)
# ============================================================================
print("=" * 70)
print("STEP 1: LOAD CLINICAL BASE (all patients)")
print("=" * 70)

clinical = pd.read_csv(CLIN_FILE, index_col=0)
print(f"Clinical: {clinical.shape}")
print(f"Sample index: {clinical.index[:3].tolist()}")

STEP 1: LOAD CLINICAL BASE (all patients)
Clinical: (1098, 144)
Sample index: ['TCGA-BH-A0W3', 'TCGA-AR-A24V', 'TCGA-E9-A1NE']


## Step 2 — Load Outcome

In [7]:
# ============================================================================
# STEP 2: LOAD OUTCOME
# ============================================================================
print("\n" + "=" * 70)
print("STEP 2: LOAD OUTCOME")
print("=" * 70)

outcome = None
for cand in OUTCOME_CANDIDATES:
    if cand.exists():
        outcome = pd.read_csv(cand, index_col=0)
        outcome.index = [
            "-".join(str(i).replace(".", "-").split("-")[:3])
            for i in outcome.index
        ]
        print(f"Loaded: {cand.name}  shape={outcome.shape}")
        break

if outcome is None:
    raise FileNotFoundError(f"outcome.csv not found. Tried: {OUTCOME_CANDIDATES}")


STEP 2: LOAD OUTCOME
Loaded: outcome.csv  shape=(1076, 2)


## Step 3 — Build Outer Join Datasets

In [9]:
# ============================================================================
# STEP 3: FOR EACH DATASET — BUILD OUTER JOIN VERSION
# ============================================================================
print("\n" + "=" * 70)
print("STEP 3: BUILD OUTER JOIN DATASETS")
print("=" * 70)
print()
print("For each dataset:")
print("  1. Read inner-join file to get selected feature names per modality")
print("  2. For each modality: load original file, select those features")
print("  3. Outer join everything onto clinical base")
print("  4. Add MODALITY_missing indicator (1 = patient had no data)")
print("  5. Fill NaN with 0")
print("  6. Join outcome (OS, OS.time)")
print()

summary = []

for ds_name in DATASET_NAMES:
    inner_file = MERGE_DIR / f"{ds_name}.csv"
    if not inner_file.exists():
        print(f"Skipping {ds_name} — not found in MERGE/")
        continue

    print(f"{'─'*70}")
    print(f"[{ds_name}]")

    # Load inner-join file to extract which features were selected
    inner = pd.read_csv(inner_file, index_col=0)
    n_inner = len(inner)

    # Get feature lists per modality from column names
    modality_features = {}
    for mod_name, prefix in MODALITY_PREFIXES.items():
        cols = [c for c in inner.columns if c.startswith(prefix)]
        if cols:
            modality_features[mod_name] = cols
            # Strip prefix to get original feature names
            modality_features[mod_name + "_original"] = [
                c[len(prefix):] for c in cols
            ]

    print(f"  Inner join had: {n_inner} samples")
    for mod_name, prefix in MODALITY_PREFIXES.items():
        n = len(modality_features.get(mod_name, []))
        print(f"  {mod_name:6s}: {n} features")

    # Start with clinical base
    merged = clinical.copy()
    print(f"\n  Building outer join:")
    print(f"    Clinical base: {len(merged)} patients")

    # For each modality: find the statistical_filtered file and outer join
    for mod_name, prefix in MODALITY_PREFIXES.items():
        if mod_name not in modality_features:
            continue

        selected_cols_prefixed  = modality_features[mod_name]
        selected_cols_original  = modality_features[mod_name + "_original"]

        # Find the modality data from the inner-join file itself
        # (re-extract original values — they're already there, just outer join)
        mod_data = inner[selected_cols_prefixed].copy()

        # Outer join: align on index
        merged = merged.join(mod_data, how="outer")

        # Add missingness indicator
        missing_indicator = mod_data.isnull().all(axis=1).astype(int)
        # For patients not in inner join at all, they're missing entirely
        missing_col = f"{prefix}missing"
        merged[missing_col] = 0
        # Patients in merged but not in mod_data are missing
        missing_patients = merged.index.difference(mod_data.index)
        merged.loc[missing_patients, missing_col] = 1

        # Fill NaN with 0 for this modality
        merged[selected_cols_prefixed] = merged[selected_cols_prefixed].fillna(0)

        n_with = (merged[missing_col] == 0).sum()
        n_without = (merged[missing_col] == 1).sum()
        print(f"    + {mod_name:6s}: {n_with} with data, {n_without} missing → indicator added")

    # Fill any remaining NaN in clinical features with 0
    # (clinical is the base so shouldn't have many, but just in case)
    clin_cols = [c for c in merged.columns if c.startswith("CLIN_")]
    merged[clin_cols] = merged[clin_cols].fillna(0)

    # Join outcome
    merged = merged.join(outcome[["OS", "OS.time"]], how="inner")

    print(f"\n  Final shape: {merged.shape}")
    print(f"  Samples: inner={n_inner} → outer={len(merged)}  "
          f"(+{len(merged)-n_inner} patients recovered)")
    print(f"  Events (OS=1): {int(merged['OS'].sum())}")

    # Save
    out_file = OUTPUT_DIR / f"{ds_name}.csv"
    merged.to_csv(out_file)
    print(f"  Saved: {out_file.name}")

    summary.append({
        "dataset":          ds_name,
        "n_inner":          n_inner,
        "n_outer":          len(merged),
        "n_recovered":      len(merged) - n_inner,
        "n_events":         int(merged["OS"].sum()),
        "n_features":       merged.shape[1] - 2,  # minus OS, OS.time
        "n_missing_inds":   len(MODALITY_PREFIXES),
    })


STEP 3: BUILD OUTER JOIN DATASETS

For each dataset:
  1. Read inner-join file to get selected feature names per modality
  2. For each modality: load original file, select those features
  3. Outer join everything onto clinical base
  4. Add MODALITY_missing indicator (1 = patient had no data)
  5. Fill NaN with 0
  6. Join outcome (OS, OS.time)

──────────────────────────────────────────────────────────────────────
[01_ultra_conservative]
  Inner join had: 526 samples
  RNA   : 50 features
  CNV   : 50 features
  MUT   : 50 features
  PROT  : 50 features
  METH  : 50 features
  MIRNA : 50 features

  Building outer join:
    Clinical base: 1098 patients
    + RNA   : 526 with data, 572 missing → indicator added
    + CNV   : 526 with data, 572 missing → indicator added
    + MUT   : 526 with data, 572 missing → indicator added
    + PROT  : 526 with data, 572 missing → indicator added
    + METH  : 526 with data, 572 missing → indicator added
    + MIRNA : 526 with data, 572 missing

## Step 4 — Summary

In [11]:
# ============================================================================
# STEP 4: SUMMARY
# ============================================================================
print("\n" + "=" * 70)
print("STEP 4: SUMMARY")
print("=" * 70)

if summary:
    df_sum = pd.DataFrame(summary)
    print(df_sum.to_string(index=False))
    df_sum.to_csv(OUTPUT_DIR / "outer_join_summary.csv", index=False)

    print(f"\nAvg samples recovered: {df_sum['n_recovered'].mean():.0f} per dataset")
    print(f"Avg events: {df_sum['n_events'].mean():.0f}")
    print(f"\nAll datasets saved to: {OUTPUT_DIR}")
    print(f"Original MERGE/ datasets unchanged.")
    print(f"\nNext step: run MB script pointing to MERGE_outer/ instead of MERGE/")


STEP 4: SUMMARY
              dataset  n_inner  n_outer  n_recovered  n_events  n_features  n_missing_inds
01_ultra_conservative      526     1076          550       150         450               6
      02_conservative      526     1076          550       150         450               6
          03_standard      526     1076          550       150         450               6
   04_fdr_significant      526     1076          550       150         450               6
          05_balanced      526     1076          550       150         450               6
       06_correlation      526     1076          550       150         450               6
    07_top_correlated      526     1076          550       150         450               6
         08_composite      526     1076          550       150         450               6

Avg samples recovered: 550 per dataset
Avg events: 150

All datasets saved to: C:\Users\olegk\Desktop\Thesis_v3\01_Causal_feature_extraction\MERGE_outer
Original M